In [1]:
! pip install together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.8 MB/s eta 0:00:00


In [8]:
import json
from together import Together
import requests
from googletrans import Translator

class CustomSmartHomeAgent:
    # def __init__(self, llm_api_key="6ba1bc9d36c18b34a8e1a820d132b0b3614192ca291165e56799eff76f03304b", primary_llm_language='en'):
    def __init__(self, llm_api_key="4e2965a15e4f5300c63125a3f8a453e501fab786da4509ccd9463fabf107107f", primary_llm_language='en'):

        """
        Initializes the Smart Home Agent.
        llm_api_key: Your API key for the chosen LLM service (TogetherAI).
        primary_llm_language: The language your core LLM prompts are optimized for (e.g., 'en').
        """
        self.llm_api_key = llm_api_key
        self.primary_llm_language = primary_llm_language
        self.llm_client = Together(api_key=self.llm_api_key)
        print(f"CustomSmartHomeAgent initialized. LLM will primarily process in: {self.primary_llm_language}")

    def _detect_language(self, text):
        if any('\u0600' <= char <= '\u06FF' for char in text):
            return 'fa'
        return 'en'

    def _translate_text(self, text, source_lang, target_lang):
        if source_lang == target_lang:
            return text
        # print(f"[LibreTranslate] Translating '{text}' from {source_lang} to {target_lang}")
        try:
            # resp = requests.post(
            #     "https://libretranslate.de/translate",
            #     data={
            #         "q": text,
            #         "source": source_lang,
            #         "target": target_lang,
            #         "format": "text"
            #     },
            #     timeout=10
            # )
            # resp.raise_for_status()
            # translated = resp.json()["translatedText"]
            # print(translated)
            # return translated
            translator = Translator()
            result = translator.translate(text, src=source_lang, dest=target_lang)
            return result.text

        except Exception as e:
            print(f"Translation error: {e}")
            return f"TranslationError_{text}_to_{target_lang}"

    def _get_llm_analysis(self, command_text_for_llm):
        prompt = f"""
        You are an AI assistant for a smart home. Analyze the user's command.
        Respond ONLY with a JSON object containing:
        - "intent": (e.g., "control_device", "get_weather", "get_news", "get_time", "unknown")
        - "device": (e.g., "lamp", "ac", "tv", "none")
        - "action": (e.g., "turn_on", "turn_off", "set_temperature", "change_channel", "fetch", "none")
        - "parameters": A dictionary of parameters (e.g., {{"location": "kitchen", "temperature": 22, "topic": "technology"}})

        User command: "{command_text_for_llm}"
        JSON Analysis:
        """
        print(f"\n--- Sending to LLM (in {self.primary_llm_language}) ---")
        print(prompt)
        print("--- End of Prompt ---")
        try:
            response = self.llm_client.chat.completions.create(
                model="meta-llama/Llama-3-70b-chat-hf",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                response_format={"type": "json_object"}
            )
            llm_output_str = response.choices[0].message.content
            print(f"LLM Raw Output: {llm_output_str}")
            return json.loads(llm_output_str)
        except Exception as e:
            print(f"Error calling LLM or processing response: {e}")
            return {"intent": "error", "message": str(e)}

    def execute_command(self, natural_language_command):
        original_lang = self._detect_language(natural_language_command)
        print(f"\nOriginal command: '{natural_language_command}' (Detected lang: {original_lang})")
        if original_lang != self.primary_llm_language:
            command_for_llm = self._translate_text(natural_language_command, original_lang, self.primary_llm_language)
        else:
            command_for_llm = natural_language_command
        if not command_for_llm:
            print("Error: Command became empty after translation step.")
            return self._generate_response("translation_error", original_lang, {})
        analysis = self._get_llm_analysis(command_for_llm)
        print(f"LLM Analysis: {json.dumps(analysis, indent=2)}")
        intent = analysis.get("intent")
        device = analysis.get("device")
        action = analysis.get("action")
        parameters = analysis.get("parameters", {})
        execution_result_message = "Action executed (or would be)."
        if intent == "control_device":
            if device == "lamp":
                execution_result_message = self._control_lamp(action, parameters)
            elif device == "ac":
                execution_result_message = self._control_ac(action, parameters)
            elif device == "tv":
                execution_result_message = self._control_tv(action, parameters)
            else:
                intent = "unknown_device"
                parameters["device_name"] = device
        elif intent == "get_weather":
            execution_result_message = self._get_weather(parameters)
        elif intent == "get_news":
            execution_result_message = self._get_news(parameters)
        elif intent == "get_time":
            execution_result_message = self._get_time(parameters)
        final_response = self._generate_response(intent, original_lang, parameters, execution_result_message, device, action)
        print(f"Agent Final Response ({original_lang}): {final_response}")
        return final_response

    def _generate_response(self, intent, lang, params, execution_message="N/A", device_name=None, action_name=None):
        response_text = ""
        location = params.get("location", "the specified location")
        topic = params.get("topic")
        if lang == 'en':
            if intent == "control_device":
                response_text = f"Okay, {action_name} the {device_name} in {location}. Status: {execution_message}"
            elif intent == "get_weather":
                response_text = f"Fetching weather for {location}. Result: {execution_message}"
            elif intent == "get_news":
                response_text = f"Getting news about {topic if topic else 'general topics'}. Result: {execution_message}"
            elif intent == "get_time":
                response_text = f"The current time is: {execution_message}"
            elif intent == "unknown_device":
                response_text = f"Sorry, I don't know how to control a device named '{params.get('device_name', 'unknown device')}."
            elif intent == "translation_error":
                response_text = "I had trouble understanding the command after attempting translation."
            elif intent == "error":
                response_text = f"An internal error occurred: {params.get('message', 'Please try again.')}"
            else:
                response_text = "I'm sorry, I didn't understand that command. Can you please rephrase?"
        elif lang == 'fa':
            if intent == "control_device":
                action_fa = {"turn_on": "روشن کردن", "turn_off": "خاموش کردن", "set_temperature": "تنظیم دمای"}.get(action_name, action_name)
                device_fa = {"lamp": "چراغ", "ac": "کولر", "tv": "تلویزیون"}.get(device_name, device_name)
                location_fa = {"kitchen": "آشپزخانه", "Room 1": "اتاق ۱"}.get(location, location)
                response_text = f"بسیار خب، {action_fa} {device_fa} در {location_fa}. وضعیت: {self._translate_text(execution_message, 'en', 'fa')}"
            elif intent == "get_weather":
                response_text = f"درحال دریافت اطلاعات آب و هوا برای {location}. نتیجه: {self._translate_text(execution_message, 'en', 'fa')}"
            elif intent == "get_news":
                response_text = f"درحال دریافت اخبار در مورد {topic if topic else 'موضوعات عمومی'}. نتیجه: {self._translate_text(execution_message, 'en', 'fa')}"
            elif intent == "get_time":
                response_text = f"ساعت فعلی: {self._translate_text(execution_message, 'en', 'fa')}"
            elif intent == "unknown_device":
                response_text = f"متاسفم، من نمی دانم چگونه دستگاهی به نام '{params.get('device_name', 'دستگاه ناشناس')}' را کنترل کنم."
            elif intent == "translation_error":
                response_text = "پس از تلاش برای ترجمه، در درک دستور مشکل داشتم."
            elif intent == "error":
                response_text = f"یک خطای داخلی رخ داد: {params.get('message', 'لطفا دوباره تلاش کنید.')}"
            else:
                response_text = "متاسفم، متوجه دستور شما نشدم. ممکن است دوباره بیان کنید؟"
        return response_text if response_text else "Response generation error."

    def _control_lamp(self, action, params):
        location = params.get("location", "default_location")
        return f"Lamp at {location} action {action} processed."

    def _control_ac(self, action, params):
        location = params.get("location", "default_location")
        temperature = params.get("temperature")
        return f"AC at {location} action {action} (temp: {temperature}) processed."

    def _control_tv(self, action, params):
        location = params.get("location", "living_room")
        channel = params.get("channel_name")
        return f"TV at {location} action {action} (channel: {channel}) processed."

    # def _get_weather(self, params):
    #     location = params.get("location", "current_city")
    #     return f"Weather for {location} is sunny, 25°C (mock)."

    def _get_news(self, params):
        topic = params.get("topic", "general")
        return f"Top news for {topic}: AI is amazing (mock)."

    def _get_time(self, params):
        from datetime import datetime
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _get_weather(self, params):
        location = params.get("location", "Tehran")
        # Use Open-Meteo geocoding to get latitude and longitude
        try:
            geo_resp = requests.get(
                "https://geocoding-api.open-meteo.com/v1/search",
                params={"name": location, "count": 1, "language": "en", "format": "json"}
            )
            geo_data = geo_resp.json()
            if "results" not in geo_data or not geo_data["results"]:
                return f"Could not find location '{location}'."
            lat = geo_data["results"][0]["latitude"]
            lon = geo_data["results"][0]["longitude"]
            # Get current weather
            weather_resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": lat,
                    "longitude": lon,
                    "current_weather": True
                }
            )
            weather_data = weather_resp.json()
            if "current_weather" in weather_data:
                temp = weather_data["current_weather"]["temperature"]
                wind = weather_data["current_weather"]["windspeed"]
                weather_str = f"Weather for {location}: {temp}°C, wind {wind} km/h."
                return weather_str
            else:
                return f"Could not get weather for '{location}'."
        except Exception as e:
            return f"Error fetching weather: {e}"



In [9]:
if __name__ == "__main__":
    # agent = CustomSmartHomeAgent(llm_api_key="6ba1bc9d36c18b34a8e1a820d132b0b3614192ca291165e56799eff76f03304b")
    agent = CustomSmartHomeAgent(llm_api_key="4e2965a15e4f5300c63125a3f8a453e501fab786da4509ccd9463fabf107107f")

    # commands_to_test = [
    #    "هوای مشهد"
    # ]

    commands_to_test = [
       " تاریخ امروز چیست؟"
    ]

    for cmd in commands_to_test:
        response = agent.execute_command(cmd)
        print("="*30)

CustomSmartHomeAgent initialized. LLM will primarily process in: en

Original command: ' تاریخ امروز چیست؟' (Detected lang: fa)

--- Sending to LLM (in en) ---

        You are an AI assistant for a smart home. Analyze the user's command.
        Respond ONLY with a JSON object containing:
        - "intent": (e.g., "control_device", "get_weather", "get_news", "get_time", "unknown")
        - "device": (e.g., "lamp", "ac", "tv", "none")
        - "action": (e.g., "turn_on", "turn_off", "set_temperature", "change_channel", "fetch", "none")
        - "parameters": A dictionary of parameters (e.g., {"location": "kitchen", "temperature": 22, "topic": "technology"})

        User command: "What is the date of today?"
        JSON Analysis:
        
--- End of Prompt ---
LLM Raw Output: {
"intent": "get_time",
"device": "none",
"action": "fetch",
"parameters": {}
}
LLM Analysis: {
  "intent": "get_time",
  "device": "none",
  "action": "fetch",
  "parameters": {}
}
Agent Final Response (fa):

In [ ]:
from googletrans import Translator

def googletrans_translate(text, src='en', dest='fa'):
    translator = Translator()
    result = translator.translate(text, src=src, dest=dest)
    return result.text

# Example:
print(googletrans_translate("آشپزخانه", src='fa', dest='en'))

Kitchen


In [5]:
%pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.8 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=d2d78e354247f6b4bea788b75a21f6fcf41be443a8bb222d68892a5e9133a135
  Stored in directory: /root/.cache/pip/wheels/39/17/6f/66a045ea3d168826074691b4b787b8f324d3f646d755443fda
Successfully built googletrans
  Attempting uninstall: hyperframe
    Found existing installation: hyperframe 6.1.0
    Uninstalling hyperfram